# PyTorch Basics: Neural Network Basics

This notebook covers:
- The nn.Module class
- Building neural networks
- Common layers (Linear, Conv2d, etc.)
- Activation functions
- Loss functions
- Forward pass

## The nn.Module Class

All neural networks in PyTorch inherit from `nn.Module`.

```
┌─────────────────────────────────────┐
│         nn.Module                   │
├─────────────────────────────────────┤
│  • __init__()  - define layers      │
│  • forward()   - define computation │
│  • parameters() - get all params    │
│  • train()     - set training mode  │
│  • eval()      - set eval mode      │
└─────────────────────────────────────┘
```


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")


PyTorch version: 2.8.0


## 1. Your First Neural Network

Let's build a simple feedforward network for MNIST (28x28 images → 10 classes).


In [2]:
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        # Define layers
        self.fc1 = nn.Linear(784, 128)  # 28*28 = 784 input features
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)    # 10 output classes
    
    def forward(self, x):
        # Flatten image
        x = x.view(-1, 784)
        
        # Layer 1
        x = self.fc1(x)
        x = F.relu(x)
        
        # Layer 2
        x = self.fc2(x)
        x = F.relu(x)
        
        # Layer 3 (no activation - we'll use CrossEntropyLoss)
        x = self.fc3(x)
        return x

# Create model
model = SimpleNet()
print(model)
print()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")


SimpleNet(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)

Total parameters: 109,386


## 2. Common Layers

PyTorch provides many built-in layers.

### Linear (Fully Connected) Layers


In [3]:
# Linear layer: y = xW^T + b
linear = nn.Linear(in_features=10, out_features=5)
print(f"Weight shape: {linear.weight.shape}")
print(f"Bias shape: {linear.bias.shape}")
print()

# Test forward pass
x = torch.randn(3, 10)  # batch_size=3, features=10
output = linear(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")


Weight shape: torch.Size([5, 10])
Bias shape: torch.Size([5])

Input shape: torch.Size([3, 10])
Output shape: torch.Size([3, 5])


### Convolutional Layers

Used for images and spatial data.

```
Input Image → Conv Layer → Feature Map

   [3x32x32]  →  Conv2d(3,16,3)  →  [16x30x30]
   (RGB)          (16 filters)        (features)
```


In [4]:
# Conv2d: applies 2D convolution
conv = nn.Conv2d(
    in_channels=3,      # RGB
    out_channels=16,    # 16 filters
    kernel_size=3,      # 3x3 kernel
    stride=1,
    padding=1
)

print(f"Weight shape: {conv.weight.shape}")  # (out_ch, in_ch, kh, kw)
print(f"Bias shape: {conv.bias.shape}")
print()

# Test forward pass
x = torch.randn(1, 3, 32, 32)  # (batch, channels, height, width)
output = conv(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")


Weight shape: torch.Size([16, 3, 3, 3])
Bias shape: torch.Size([16])

Input shape: torch.Size([1, 3, 32, 32])
Output shape: torch.Size([1, 16, 32, 32])


### Pooling Layers

Reduce spatial dimensions.

```
[4x4] → MaxPool(2x2) → [2x2]

1 2 3 4        6  8
5 6 7 8   →   14 16
9 10 11 12
13 14 15 16
```


In [5]:
# MaxPool2d: takes maximum value in window
maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

x = torch.randn(1, 16, 32, 32)
output = maxpool(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print()

# AvgPool2d: takes average
avgpool = nn.AvgPool2d(kernel_size=2, stride=2)
output = avgpool(x)
print(f"After average pooling: {output.shape}")


Input shape: torch.Size([1, 16, 32, 32])
Output shape: torch.Size([1, 16, 16, 16])

After average pooling: torch.Size([1, 16, 16, 16])


### Activation Functions

Add non-linearity to the network.

```
Common Activations:
┌──────────────────────────────────────────┐
│ ReLU(x) = max(0, x)                      │
│ Sigmoid(x) = 1 / (1 + e^(-x))           │
│ Tanh(x) = (e^x - e^(-x)) / (e^x + e^(-x))│
│ LeakyReLU(x) = max(0.01x, x)            │
└──────────────────────────────────────────┘
```


In [6]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

print(f"Input: {x}")
print(f"ReLU: {F.relu(x)}")
print(f"Sigmoid: {torch.sigmoid(x)}")
print(f"Tanh: {torch.tanh(x)}")
print(f"LeakyReLU: {F.leaky_relu(x, negative_slope=0.01)}")


Input: tensor([-2., -1.,  0.,  1.,  2.])
ReLU: tensor([0., 0., 0., 1., 2.])
Sigmoid: tensor([0.1192, 0.2689, 0.5000, 0.7311, 0.8808])
Tanh: tensor([-0.9640, -0.7616,  0.0000,  0.7616,  0.9640])
LeakyReLU: tensor([-0.0200, -0.0100,  0.0000,  1.0000,  2.0000])


## 3. Building a CNN for MNIST

Let's build a proper convolutional neural network.


In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        # Pooling
        self.pool = nn.MaxPool2d(2, 2)
        
        # Fully connected layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.conv1(x)))  # 28x28 → 14x14
        
        # Conv block 2
        x = self.pool(F.relu(self.conv2(x)))  # 14x14 → 7x7
        
        # Flatten
        x = x.view(-1, 64 * 7 * 7)
        
        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Create model
model = CNN()
print(model)
print()

# Test forward pass
dummy_input = torch.randn(4, 1, 28, 28)  # batch_size=4
output = model(dummy_input)
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

Input shape: torch.Size([4, 1, 28, 28])
Output shape: torch.Size([4, 10])

Total parameters: 421,642
Trainable parameters: 421,642


## 4. Loss Functions

Loss functions measure how wrong the model is.

```
Common Loss Functions:
┌─────────────────────────────────────────┐
│ MSELoss - Mean Squared Error            │
│   L = mean((y_pred - y_true)²)          │
│                                          │
│ CrossEntropyLoss - Classification        │
│   L = -log(softmax(y_pred)[y_true])    │
│                                          │
│ BCELoss - Binary Cross Entropy          │
│   L = -[y*log(p) + (1-y)*log(1-p)]     │
└─────────────────────────────────────────┘
```


In [8]:
# Regression: MSE Loss
mse_loss = nn.MSELoss()
y_pred = torch.tensor([2.5, 0.0, 2.1, 7.8])
y_true = torch.tensor([3.0, -0.5, 2.0, 7.0])
loss = mse_loss(y_pred, y_true)
print(f"MSE Loss: {loss.item():.4f}")
print()

# Classification: CrossEntropyLoss
ce_loss = nn.CrossEntropyLoss()
logits = torch.randn(3, 5)  # 3 samples, 5 classes
targets = torch.tensor([1, 0, 4])  # class indices
loss = ce_loss(logits, targets)
print(f"CrossEntropy Loss: {loss.item():.4f}")
print()

# Binary Classification: BCELoss
bce_loss = nn.BCELoss()
y_pred = torch.sigmoid(torch.randn(10))  # predictions [0, 1]
y_true = torch.randint(0, 2, (10,)).float()  # binary labels
loss = bce_loss(y_pred, y_true)
print(f"BCE Loss: {loss.item():.4f}")


MSE Loss: 0.2875

CrossEntropy Loss: 1.6185

BCE Loss: 0.9460


## 5. Sequential Models

For simple architectures, use nn.Sequential.


In [9]:
# Method 1: Sequential with ordered layers
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 10)
)

print(model)
print()

# Method 2: Sequential with OrderedDict
from collections import OrderedDict

model = nn.Sequential(OrderedDict([
    ('fc1', nn.Linear(784, 128)),
    ('relu1', nn.ReLU()),
    ('dropout1', nn.Dropout(0.2)),
    ('fc2', nn.Linear(128, 64)),
    ('relu2', nn.ReLU()),
    ('dropout2', nn.Dropout(0.2)),
    ('fc3', nn.Linear(64, 10))
]))

print(model)
print()

# Access specific layer
print(f"First layer: {model.fc1}")
print(f"First layer weight shape: {model.fc1.weight.shape}")


Sequential(
  (0): Linear(in_features=784, out_features=128, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=128, out_features=64, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.2, inplace=False)
  (6): Linear(in_features=64, out_features=10, bias=True)
)

Sequential(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.2, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.2, inplace=False)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)

First layer: Linear(in_features=784, out_features=128, bias=True)
First layer weight shape: torch.Size([128, 784])


## 📝 Summary

✅ Built neural networks with nn.Module  
✅ Used common layers (Linear, Conv, Pool)  
✅ Applied activation functions  
✅ Implemented CNN architecture  
✅ Understood loss functions  
✅ Created models with nn.Sequential  

### Key Concepts

1. **Always call `super().__init__()`** in `__init__()`
2. **Define layers in `__init__`**, use them in `forward()`
3. **No need to call backward()** on loss - optimizer handles it
4. **Use ReLU** as default activation (works well in practice)

### Next Steps

- **Next Tutorial**: `05_training_basics.ipynb` - Train your first model!
- **Practice**: Build different architectures
- **Challenge**: Implement ResNet skip connections

### Additional Resources

- [PyTorch nn Documentation](https://pytorch.org/docs/stable/nn.html)
- [PyTorch Module Tutorial](https://pytorch.org/tutorials/beginner/nn_tutorial.html)


## 🎯 Practice Exercises


In [10]:
# Exercise 1: Build a 3-layer MLP with BatchNorm
# Your code here:


# Exercise 2: Implement a simple autoencoder
# Your code here:


# Exercise 3: Create a custom activation function
# Your code here:


# Exercise 4: Build a network that takes two inputs and concatenates them
# Your code here:
